In [ ]:
from google import genai
from google.genai import types

# Define the function declaration for the model
schedule_meeting_function = {
    "name": "schedule_meeting",
    "description": "Schedules a meeting with specified attendees at a given time and date.",
    "parameters": {
        "type": "object",
        "properties": {
            "attendees": {
                "type": "array",
                "items": {"type": "string"},
                "description": "List of people attending the meeting.",
            },
            "date": {
                "type": "string",
                "description": "Date of the meeting (e.g., '2024-07-29')",
            },
            "time": {
                "type": "string",
                "description": "Time of the meeting (e.g., '15:00')",
            },
            "topic": {
                "type": "string",
                "description": "The subject or topic of the meeting.",
            },
        },
        "required": ["attendees", "date", "time", "topic"],
    },
}

# Second Function 
get_weather_function = {
    "name": "get_weather",
    "description": "Get the weather forecast for a specific location and date.",
    "parameters": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "The location for which to get the weather forecast (e.g., 'New York City').",
            },
            "date": {
                "type": "string",
                "description": "The date for which to get the weather forecast (e.g., '2024-07-29').",
            }
        },
        "required": ["location", "date"],
    }
}

# Configure the client and tools
client = genai.Client(
    api_key="your_api_key_here"
)
tools = types.Tool(function_declarations=[get_weather_function, schedule_meeting_function])
config = types.GenerateContentConfig(tools=[tools])

contents = [
    types.Content(
        role="user", parts=[types.Part(text="What's the weather forecast for New York City on July 29, 2024?")]
    )
]

response = client.models.generate_content(
    model="gemini-3.1-pro-preview",
    contents=contents,
    config=config,
)

print(response.candidates[0].content.parts[0].function_call)

In [ ]:
# Extract tool call details, it may not be in the first part.
tool_call = response.candidates[0].content.parts[0].function_call

if tool_call.name == "get_weather":
    result = "The weather forecast for New York City on July 29, 2024, is expected to be sunny with a high of 999°F and a low of 700°F."
    print(f"Function execution result: {result}")

In [ ]:
# Create a function response part
function_response_part = types.Part.from_function_response(
    name=tool_call.name,
    response={"result": result},
)

# Append function call and result of the function execution to contents
contents.append(response.candidates[0].content) # Append the content from the model's response.
contents.append(types.Content(role="user", parts=[function_response_part])) # Append the function response

client = genai.Client(
    api_key="your_api_key_here"
)
final_response = client.models.generate_content(
    model="gemini-3.1-pro-preview",
    config=config,
    contents=contents,
)

print(final_response.text)

### ⼿动模拟⼯具调⽤

GEMINI Example

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(
    api_key="your_api_key_here"
)

contents = [
    types.Content(
        role="user", parts=[types.Part(text="""Please extract the recipe from the following text.
The user wants to make delicious chocolate chip cookies.
They need 2 and 1/4 cups of all-purpose flour, 1 teaspoon of baking soda,
1 teaspoon of salt, 1 cup of unsalted butter (softened), 3/4 cup of granulated sugar,
3/4 cup of packed brown sugar, 1 teaspoon of vanilla extract, and 2 large eggs.
For the best part, they'll need 2 cups of semisweet chocolate chips.
First, preheat the oven to 375°F (190°C). Then, in a small bowl, whisk together the flour,
baking soda, and salt. In a large bowl, cream together the butter, granulated sugar, and brown sugar
until light and fluffy. Beat in the vanilla and eggs, one at a time. Gradually beat in the dry
ingredients until just combined. Finally, stir in the chocolate chips. Drop by rounded tablespoons
onto ungreased baking sheets and bake for 9 to 11 minutes.""")]
    )
]

system_prompt = """
You must return valid HTML only.
Do not wrap in markdown.
Do not add explanations.
Schema:
<h3>Recipe Name</h3>
<ul>
<li>ingredient name: quantity</li>
</ul>
<ul>
<li>instruction 1</li>
<li>instruction 2</li>
</ul>

"""

response = client.models.generate_content(
    model="gemini-3.1-pro-preview",
    contents=contents,
    config=types.GenerateContentConfig(system_instruction=system_prompt),
)

print(response.text)

<h3>Chocolate Chip Cookies</h3>
<ul>
<li>all-purpose flour: 2 and 1/4 cups</li>
<li>baking soda: 1 teaspoon</li>
<li>salt: 1 teaspoon</li>
<li>unsalted butter (softened): 1 cup</li>
<li>granulated sugar: 3/4 cup</li>
<li>packed brown sugar: 3/4 cup</li>
<li>vanilla extract: 1 teaspoon</li>
<li>large eggs: 2</li>
<li>semisweet chocolate chips: 2 cups</li>
</ul>
<ul>
<li>Preheat the oven to 375°F (190°C).</li>
<li>In a small bowl, whisk together the flour, baking soda, and salt.</li>
<li>In a large bowl, cream together the butter, granulated sugar, and brown sugar until light and fluffy.</li>
<li>Beat in the vanilla and eggs, one at a time.</li>
<li>Gradually beat in the dry ingredients until just combined.</li>
<li>Finally, stir in the chocolate chips.</li>
<li>Drop by rounded tablespoons onto ungreased baking sheets and bake for 9 to 11 minutes.</li>
</ul>


OpenAI Example

In [6]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio"
)

model="omnicoder-9b"

response = client.responses.create(
    model=model,
    input="""
You must return valid HTML only.
Do not wrap in markdown.
Do not add explanations.
Schema:
<h3>Recipe Name</h3>
<ul>
<li>ingredient name: quantity</li>
</ul>
<ul>
<li>instruction 1</li>
<li>instruction 2</li>
</ul>    


Please extract the recipe from the following text.
The user wants to make delicious chocolate chip cookies.
They need 2 and 1/4 cups of all-purpose flour, 1 teaspoon of baking soda,
1 teaspoon of salt, 1 cup of unsalted butter (softened), 3/4 cup of granulated sugar,
3/4 cup of packed brown sugar, 1 teaspoon of vanilla extract, and 2 large eggs.
For the best part, they'll need 2 cups of semisweet chocolate chips.
First, preheat the oven to 375°F (190°C). Then, in a small bowl, whisk together the flour,
baking soda, and salt. In a large bowl, cream together the butter, granulated sugar, and brown sugar
until light and fluffy. Beat in the vanilla and eggs, one at a time. Gradually beat in the dry
ingredients until just combined. Finally, stir in the chocolate chips. Drop by rounded tablespoons
onto ungreased baking sheets and bake for 9 to 11 minutes."""
)

print(response.output_text)



<h3>Chocolate Chip Cookies</h3>
<ul>
<li>all-purpose flour: 2 and 1/4 cups</li>
<li>baking soda: 1 teaspoon</li>
<li>salt: 1 teaspoon</li>
<li>unsalted butter (softened): 1 cup</li>
<li>granulated sugar: 3/4 cup</li>
<li>packed brown sugar: 3/4 cup</li>
<li>vanilla extract: 1 teaspoon</li>
<li>eggs: 2 large</li>
<li>semisweet chocolate chips: 2 cups</li>
</ul>
<ul>
<li>Preheat the oven to 375°F (190°C).</li>
<li>In a small bowl, whisk together the flour, baking soda, and salt.</li>
<li>In a large bowl, cream together the butter, granulated sugar, and brown sugar until light and fluffy.</li>
<li>Beat in the vanilla and eggs, one at a time.</li>
<li>Gradually beat in the dry ingredients until just combined.</li>
<li>Finally, stir in the chocolate chips.</li>
<li>Drop by rounded tablespoons onto ungreased baking sheets and bake for 9 to 11 minutes.</li>
</ul>
